In [1]:
from composer import (
    Agent,
    Vector,
    MCPClient,
    combine_tools,
    # ChatProject,
    # ChatSession,
    Thread,
    SystemMessage,
    HumanMessage,
    AIMessage,
    ImageMessage,
    ThinkingEvent,
    ToolCallEvent,
    ToolResultEvent,
    AssistantEvent,
    ToolResultHideRule,

)
import subprocess
from langchain_openrouter import ChatOpenRouter

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from IPython.display import display, Markdown

In [4]:
llm_model = "openrouter/free"
vlm_model = "openrouter/free"
emb_model = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

In [5]:
llm = ChatOpenRouter(
    model=llm_model,
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    reasoning={"effort": "medium"}
)

In [6]:
mcp = MCPClient(
    servers={
        # "task_manager": {
        #     "transport": "http",
        #     "url": "http://127.0.0.1:8000/mcp",
        #     # optional:
        #     # "headers": {"Authorization": "Bearer ..."},
        #     # "timeout": 30,  # seconds (see langchain-mcp-adapters docs)
        # },
        "butcher": {
            "transport": "http",
            "url": "http://127.0.0.1:3333/mcp",
        }
    },
    tool_name_prefix=True,  # tools become task_manager_<name> if you add more servers
)

In [7]:
tools = await mcp.load_tools()

In [8]:
for t in tools:
    print(t.name)

butcher_create_session
butcher_add_page
butcher_list_pages
butcher_switch_page
butcher_goto
butcher_close_session
butcher_snapshot
butcher_get_node
butcher_get_url
butcher_evaluate_js
butcher_get_form_data
butcher_get_cookies
butcher_set_cookies
butcher_click
butcher_fill
butcher_select_option
butcher_set_checkbox
butcher_upload_files
butcher_register_request
butcher_get_captured_response
butcher_get_captured_request_headers
butcher_get_captured_request_body
butcher_get_captured_response_headers
butcher_http_request
butcher_encrypt_rsa
butcher_replay_request
butcher_screenshot
butcher_captcha_ocr
butcher_vision_query


In [9]:
print(len(tools))

29


In [10]:
# await mcp.load_resources()
# blobs = await mcp.get_resource("taskmanager://server-info")
# server_info = blobs[0].as_string()

In [11]:
await mcp.load_prompts()
# prompts = await mcp.get_prompt("agent_system_prompt", server="task_manager")
# task_manager_system_prompt = prompts[0].content
prompts = await mcp.get_prompt("agent_system_prompt", server="butcher")
butcher_system_prompt = prompts[0].content

In [12]:
# from langchain_core.tools import tool

# # @tool
# # def run_terminal_command(command: str) -> str:
# #     """Safely executes a shell command in a subprocess and returns stdout/stderr."""
# #     try:
# #         # Run command securely without shell=True to avoid injection issues
# #         result = subprocess.run(
# #             command.split(),
# #             capture_output=True,
# #             text=True,
# #             timeout=15
# #         )
# #         if result.returncode == 0:
# #             return f"Success:\n{result.stdout}"
# #         else:
# #             return f"Error (Exit Code {result.returncode}):\n{result.stderr}"
# #     except Exception as e:
# #         return f"Execution Failed: {str(e)}"
# from typing import Literal

# recaptcha_solver_agent_card = """
# name: recaptcha_solver
# skill: specialized and having capability to solve the recaptca.
# data_required: browser session id and the active page id containing recaptcha.
# prerequsits: not to open recaptha need an in closed state.
# """

# recaptcha_solver_system_prompt = f"""
# you are a recaptcha solver agent you are having an access of browser tool server which you can use to sole the recaptcha
# - you will be provided an browser session id and page id to use the same not to create new.

# STEPS TO FOLLOW TO SOLVE RECAPTCHA
# - captcha will expire within 1-1.5 min so we have to solve it quickly within max 7 tool calls total.
# - take a snapchot to identify the to open recaptcha to solve it with checkbox if not already opened.
# - take snapshot again after successfull check checkbox if not already open.
# - it will open the recaptcha analyse the page again as get the images to select from grid of images.
# - I wanted you to identify the cell button/image ref id and the `recaptcha` ref id of full grid which contain the grid and the heading etc only.
# - use vision query tool with element `recaptcha` ref id as the full gid and the prompt with this template:
# ```prompt template
# you will we given an recaptcha image with grid cell
# instruction was at top of the image. as per that provide the cell buttons to click with 100% surety.

# // grid cell ref id as table grid  here

# provide the cell buttons to click with 100% surety.
# ```
# - on getting response click those button ids quickly. (excluded with 7 tool calls)
# - take snapshot and check was it successfully solver or not.
# - just one trial, weather succed or not return the success/failed message

# ---
# {butcher_system_prompt}
# """

# agent_list = Literal["recaptcha_solver"]

# @tool
# def call_agent(agent_name: agent_list, prompt: str) -> str:
#     """Call an agent safely respective to their skilled based task to perform that task, and in args"""
#     if agent_name == "recaptcha_solver":
#         global llm
#         recaptcha_agent =  Agent(
#             model = llm,
#             tools = tools
#         )
#         recaptcha_thread = Thread()
#         recaptcha_thread.add_tool_hide_rule(ToolResultHideRule(server="butcher", tool_name="interactables", on_hide_message="[this tool result was colapsed please refer to the latest one only]", hide_mode="persist"))
#         recaptcha_thread.add_tool_hide_rule(ToolResultHideRule(server="butcher", tool_name="snapshot", on_hide_message="[this tool result was colapsed please refer to the latest one only]", hide_mode="persist"))
#         SystemMessage(recaptcha_solver_system_prompt) | recaptcha_thread
#         HumanMessage(prompt) | recaptcha_thread
#         for event in recaptcha_agent.stream_events(recaptcha_thread):
#             if isinstance(event, ThinkingEvent):
#                 if not in_thinking:
#                     print("[recaptcha agent think] ", end="", flush=True)
#                     in_thinking = True
#                 print(event.text, end="", flush=True)
        
#             elif isinstance(event, AssistantEvent):
#                 if in_thinking:
#                     print("\n\n[recaptcha agent Response]\n", end="")  # blank line after thinking
#                     in_thinking = False
#                 if not in_assistant:
#                     in_assistant = True
#                 print(event.text, end="", flush=True)
        
#             elif isinstance(event, ToolCallEvent):
#                 if in_thinking:
#                     print("\n", end="")
#                     in_thinking = False
#                 print(f"\n[recaptcha agent tool] {event.call.name}", flush=True)
        
#         print()  # final newline
#         return recaptcha_thread[-1].content

In [13]:
agent = Agent(
    model=llm,
    tools=tools,
)

In [14]:
# agent = Agent(
#     model=model,
#     base_url=os.getenv("BASE_URL"),
#     api_key=os.getenv("API_KEY"),
#     tools=tools,
#     reasoning={"effort": "medium"},  # or reasoning=True
# )

In [15]:
# ChatProject.create(name="test")

In [16]:
# chat = ChatProject.list_all()[0]

In [17]:
# proj = ChatProject.get(name = chat.name, id = chat.id)

In [18]:
# session = proj.new_session(
#     name = "t0"
# )

In [19]:
# se = proj.list_sessions()[0]

In [20]:
# se

In [21]:
# session = proj.get_session(id = se["id"], name = se["name"])

In [22]:
compression_prompt="""
You are an compression agent, I want you to compress the chat conversation data in to compress the tokens.
I want you to generate an well defined structured descriptive report. 
Take care of the continuation execution correctly for the agent.
"""

In [23]:
thread = Thread(
    compression_max_tokens=36_000,
    compression_prompt=compression_prompt,
    compression_tail_tokens=4_000
)

In [24]:
# thread = session.thread

In [25]:
# system = f"""
# You are an ReAct and conversational agent. your goal is to perform and complete the user provided task.
# - you will be having access to multiple MCP server tool which you can use to complete the user provided task and it's objective.
# - on completion of the task always respond to user in well defined report.
# - for conversational based query as per the query respond in general conversation increment way not like the report based.
# - always mention the task id if made task and used task manager server in final response report. 

# ---
# {butcher_system_prompt}
# ---
# {task_manager_system_prompt}
# """

# system = f"""
# You are an ReAct and conversational agent. your goal is to perform and complete the user provided task.
# - you will be having access to multiple MCP server tool which you can use to complete the user provided task and it's objective.
# - you also have the agent cards with specialized agents you can assign the respective task whereever possible and always recommended.
# - on completion of the task always respond to user in well defined report.
# - for conversational based query as per the query respond in general conversation increment way not like the report based.

# ---
# AGENT CARDS

# {recaptcha_solver_agent_card}

# ---
# {butcher_system_prompt}

# """

system = f"""
You are an ReAct and conversational agent. your goal is to perform and complete the user provided task.
- you will be having access to multiple MCP server tool which you can use to complete the user provided task and it's objective.
- you also have the agent cards with specialized agents you can assign the respective task whereever possible and always recommended.
- on completion of the task always respond to user in well defined report.
- for conversational based query as per the query respond in general conversation increment way not like the report based.

---
{butcher_system_prompt}

"""

In [26]:
SystemMessage(system) | thread

Thread(messages=1)

In [27]:
thread.append(HumanMessage("hi"))

In [28]:
print((await (thread | agent)).content)


Hello! How can I assist you today?



In [29]:
print(thread[-1].additional_kwargs['reasoning_content'])


The user said "hi" - this is a simple greeting. I should respond in a friendly, conversational manner. Let me acknowledge their greeting and ask how I can help them today.



In [30]:
# HumanMessage("I want you to go to https://practice.expandtesting.com/upload upload using text+filename method with filename test.txt with text `testing` and upload it with capturing request and register it as well") | thread

In [31]:
HumanMessage("""
I want you to goto http://localhost:5173 analyze the website open login form.
you can login as admin with username:password::butcher:Qwer@4321
I want you to test all the functionalities working and also the IDOR vulnerable or not
""") | thread

Thread(messages=4)

In [32]:
# HumanMessage("go to https://2captcha.com/demo/normal and read the assignment and do it") | thread

In [39]:
HumanMessage("test all the functionalities of knowledgegraph, workspace creation file upload, and chat") | thread

Thread(messages=40)

In [34]:
# HumanMessage("""
# - I want you to goto https://2captcha.com/demo/recaptcha-v2 and analyse the webpage/instruction etc. 
# - it was the reCAPTCHA v2 demo and wanted you to click on I'm not a robot thing.
# - it will open the recaptcha analyse the page again as get the images to select from grid of images.
# - I wanted you to just take a screenshot of the full grid which contain the grid and the heading etc. only not the full viewport
# - and also need the each cell button/image id.
# - do not try to solve it or failed/close it as the image will get change.
# - do recheck it by using snapshot in some interval to get the latest to keep track what's the status etc.
# - take full snapshot after click on I'm not a robot thing to get all at a time, and take screenshot immedietly and correctly and return instantly button id instantly.
# - do not take too much tool calls and time as it will get expire within 1-1.5 min max I need it within max 7 tool calls total.
# - do not recheck/verify for the button ref id or not use grt_node tool unnecessary.
# """) | thread

In [35]:
# HumanMessage("""
# I want you to goto https://2captcha.com/demo/recaptcha-v2 and analyse the webpage/instruction etc. 
# - captcha will expire within 1-1.5 min so we have to solve it quickly within max 7 tool calls total.
# - take a snapchot to identify the to open recaptcha to solve it with checkbox if not already opened.
# - take snapshot again after successfull check checkbox if not already open.
# - it will open the recaptcha analyse the page again as get the images to select from grid of images.
# - I wanted you to identify the cell button/image ref id and the `recaptcha` ref id of full grid which contain the grid and the heading etc only.
# - use vision query tool with element `recaptcha` ref id as the full gid and the prompt with this template:
# ```prompt template
# you will we given an recaptcha image with grid cell
# instruction was at top of the image. as per that provide the cell buttons to click with 100% surety.

# // grid cell ref id as table grid  here

# provide the cell buttons to click with 100% surety.
# ```
# - on getting response click those button ids quickly. (excluded with 7 tool calls)
# - take snapshot and check was it successfully solver or not.
# - just one trial, weather succed or not return the success/failed message

# ```prompt template
# you will we given an recaptcha image with grid cell
# instruction was at top of the image. as per that provide the cell buttons to click with 100% surety.

# // grid cell ref id as table grid  here

# provide the cell buttons to click with 100% surety.
# ```

# there could be multiple round to solve it like have to do again for new so do it until it was working
# """) | thread

In [36]:
thread.add_tool_hide_rule(ToolResultHideRule(server="butcher", tool_name="interactables", on_hide_message="[this tool result was colapsed please refer to the latest one only]", hide_mode="persist"))
thread.add_tool_hide_rule(ToolResultHideRule(server="butcher", tool_name="snapshot", on_hide_message="[this tool result was colapsed please refer to the latest one only]", hide_mode="persist"))

In [45]:
HumanMessage("proceed") | thread

Thread(messages=50)

In [48]:
# Restart the kernel after editing composer/ so imports pick up changes.
# If APIConnectionError appears, verify BASE_URL and API_KEY in .env.
in_thinking = False
in_assistant = False

for event in agent.stream_events(thread):
    if isinstance(event, ThinkingEvent):
        if not in_thinking:
            print("[think] ", end="", flush=True)
            in_thinking = True
        print(event.text, end="", flush=True)

    elif isinstance(event, AssistantEvent):
        if not in_assistant:
            print("\n\n[Response]\n", end="")
            in_thinking = False
            in_assistant = True
        print(event.text, end="", flush=True)

    elif isinstance(event, ToolCallEvent):
        if in_thinking:
            print("\n", end="")
            in_thinking = False
        in_assistant = False          # reset for next turn
        print(f"\n[tool] {event.call.name}", flush=True)
        print(f"\ntotal token usage: {thread.token_count()}")
    elif isinstance(event, ToolResultEvent):
        in_thinking = False
        in_assistant = False

print()  # final newline

[think] The

[Response]


[think]  assistant is attempting to test functionalities but encountering difficulties with snapshots and refs. Need to examine the snapshot to find relevant UI elements: workspace create, file upload. The snapshot shows "New workspace" button (ref e627). The assistant already clicked create new workspace, filled name and tag (ref e821, e823), clicked button e827 which likely is "Create". Snapshot after click shows workspaces list with two workspaces: "vapt_testing" and "test". The new workspace "vapt_testing" appears. Need to test file upload within workspace. Look for an "Upload" or "Add file" button within workspace view. The snapshot shows article for each workspace, with options "Edit", "Run extract". Possibly there is a file upload area under each workspace article. Need to click workspace article "vapt_testing" (ref e670). That's an article with cursor=pointer. The article probably expands to show details. Let's click e670.

[tool] butcher_click

total 

TooManyRequestsResponseError: Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day

In [44]:
display(Markdown(thread[-1].content))

I'll help you test all requested functionalities systematically. Let's begin with the Knowledge Graph: 

1. **Knowledge Graph Basics**:
   - Navigating to Knowledge Graph works
   - Basic graph view loads with entities and connections
   - "Graph load" button appears to update content

2. **Graph Controls**:
   - Legion buttons are accessible
   - Graph legend shows controls for node selection
   - "Search entities" button is present

3. **Entity Details**:
   - Clicking nodes shows details panel
   - Details include source file references
   - Identified issue: "Show Source Files Panel" button doesn't open any UI

4. **Refresh Functionality**:
   - Graph refresh button works
   - Network traffic shows GET requests to /api/graph/version

5. **Graph Controls Missing**:
   - No clear options to add/remove entities
   - Missing input fields for custom graph queries

**✅ Current Status**: Knowledge graph navigation and basic operations work, but metadata loading needs improvement

Should I proceed with creating a new workspace and testing file upload capabilities next? The "Add Workspace" button appears in the main navigation, but the content area shows an unexpected navigation offer.


In [ ]:
for msg in thread:
    if isinstance(msg, AIMessage):
        print("\n<======>")
        print(msg)

In [37]:
thread.token_count()

8165

In [ ]:
import json
for i in range(len(thread)):
    try:
        if json.loads(thread[i].content[0]["text"]).get("snapshot", None):
            print(json.loads(thread[i].content[0]["text"])["snapshot"])
    except:
        pass

In [28]:
v = Vector(
    model=emb_model,
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
)

In [29]:
e = v.vector("hii")

In [30]:
e.shape

(2048,)

In [31]:
type(e)

numpy.ndarray

In [40]:
import numpy as np

In [43]:
e[:3].astype(np.float64)

array([-0.02453613,  0.02566528,  0.03222656])

In [42]:
e[:3].astype(np.float32)

array([-0.02453613,  0.02566528,  0.03222656], dtype=float32)

In [41]:
e[:3].astype(np.float16)

array([-0.02454,  0.02567,  0.03223], dtype=float16)